In [1]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install peft
!pip install accelerate
!pip install bitsandbytes
!pip install sentencepiece

In [2]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")

In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = ds["train"].to_pandas()

train_df, test_df = train_test_split(df, test_size=0.5, stratify=df["intent"])

from datasets import Dataset, DatasetDict
ds_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[:100]),
    "test": Dataset.from_pandas(test_df[:100])
})

print(ds_dataset)

DatasetDict({
    train: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__'],
        num_rows: 100
    })
    test: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__'],
        num_rows: 100
    })
})


In [4]:
ds_dataset = ds_dataset.remove_columns(["tags", "category", "intent", "__index_level_0__"])

In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the pre-trained T5 small model and tokenizer
model_name = "t5-small"  # You can choose other sizes like t5-base, t5-large, etc.
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [6]:
# Sample a smaller subset of the dataset (for example, 10% of the original dataset)

# Tokenize function
def tokenize_function(examples):
    inputs = [instruction + " </s> " + response for instruction, response in zip(examples["instruction"], examples["response"])]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)

    # Create labels
    labels = model_inputs["input_ids"].copy()
    labels = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels]
    model_inputs["labels"] = labels

    return model_inputs

# Apply the tokenization to both 'train' and 'test' datasets
ds_dataset = ds_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
# Resize the tokenizer if necessary (in case you added special tokens)
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

Embedding(32100, 512)

In [8]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [9]:
from transformers import Trainer, TrainingArguments

# Set training arguments
training_args = TrainingArguments(
    output_dir="./t5-lora-finetuned",  # Directory to save model checkpoints
    num_train_epochs=3,               # Number of epochs to train
    per_device_train_batch_size=2,    # Batch size for training
    per_device_eval_batch_size=2,     # Batch size for evaluation
    weight_decay=0.01,                # Apply weight decay for regularization
    learning_rate=1e-3,
    evaluation_strategy="epoch",
    logging_dir="./logs",  # Where logs are stored
    logging_strategy="epoch",  # Log after every epoch
    save_strategy="epoch",  # Save checkpoints after every epoch
)

# Initialize the Trainer
trainer = Trainer(
    model=model,                   # Use the LoRA-adapted model
    args=training_args,                 # Training arguments
    train_dataset=ds_dataset['train'],        # Training dataset
    tokenizer=tokenizer,                # Tokenizer for the model
    eval_dataset=ds_dataset['test'],
    data_collator=data_collator,
)

/home/justin.aj/.local/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_62323/415797258.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [10]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.441000,0.017292
2,0.069200,0.005464
3,0.038400,0.003367


TrainOutput(global_step=150, training_loss=0.18287012020746868, metrics={'train_runtime': 142.5307, 'train_samples_per_second': 2.105, 'train_steps_per_second': 1.052, 'total_flos': 40602540441600.0, 'train_loss': 0.18287012020746868, 'epoch': 3.0})